# DuckDB 分钟数据 vs `T_mindf.csv`

本 notebook 比较 `data/database/treasury_futures.duckdb/main_minute_continuous` 与
`data/csv/T_mindf.csv`。重点不只是逐行检查，还会复算因子依赖的
A/B/C/D、D1/D2/D3、开盘五分钟和尾盘两分钟特征，定位哪些交易日可能
因分钟行情变化而产生不同信号。

对比原则：

- 以分钟时间戳做 outer join，分别保留 DuckDB-only、CSV-only 和共同分钟；
- `amt/total_turnover` 先按 OHLC 自动识别 1 或 10,000 的单位乘数，再比较分钟 VWAP；
- 价格字段用 bp 展示差异，成交量和持仓量保留绝对差异；
- 信号敏感诊断复算反转等级、D 段一致性、尾盘持仓加速方向与持仓主导类型；
- 结果写到 `reports/data_pipeline/minute_data_comparison.xlsx`，逐分钟异常另存 Parquet。


In [ ]:
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for candidate in _PROJECT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root with pyproject.toml was not found.")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from datapipeline.paths import (
    DATA_PIPELINE_REPORT_DIR,
    DB_PATH,
    HISTORICAL_MINUTE_CSV,
    PROJECT_ROOT as PACKAGE_DIR,
)

CSV_PATH = HISTORICAL_MINUTE_CSV
DB_TABLE = "main_minute_continuous"

# Optional date range; None means no boundary.
DATE_FROM = None  # 例如 "2024-01-01"
DATE_TO = None    # 例如 "2026-06-11"

# 逐字段匹配容差。
PRICE_TOL = 1e-9
VWAP_TOL = 1e-9
VOLUME_TOL = 0.0
OI_TOL = 0.0

# 分钟钻取日期；None 时自动选择异常分数最高的一天。
INSPECT_DATE = None  # 例如 "2025-01-02"
EXPORT_RESULTS = True

SEGMENTS = {
    "A": ("00:00", "13:59"),
    "B": ("14:00", "14:24"),
    "C": ("14:25", "14:49"),
    "D": ("14:50", "15:15"),
    "D1": ("14:50", "14:58"),
    "D2": ("14:59", "15:06"),
    "D3": ("15:07", "15:15"),
    "open5_pre": ("09:15", "09:19"),
    "open5_post": ("09:30", "09:34"),
    "last2": ("15:14", "15:15"),
}
POST_OPEN_SWITCH = pd.Timestamp("2020-07-20")

if not CSV_PATH.is_file():
    raise FileNotFoundError(CSV_PATH)

print(f"PACKAGE_DIR={PACKAGE_DIR}")
print(f"DB={DB_PATH.name} | CSV={CSV_PATH.relative_to(PACKAGE_DIR)}")


## 1. 读取并统一字段、成交额单位


In [ ]:
def normalize_contract(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(".CFX", ".CFE", regex=False)
    )


def infer_amount_multiplier(frame: pd.DataFrame) -> tuple[float, float]:
    reference = frame[["open", "high", "low", "close"]].mean(axis=1)
    raw_vwap = frame["amount_raw"] / frame["volume"].replace(0, np.nan)
    ratio = (raw_vwap / reference).replace([np.inf, -np.inf], np.nan)
    ratio = ratio[ratio.gt(0)].dropna()
    if ratio.empty:
        raise ValueError("无法由 amount/volume 与 OHLC 推断成交额单位")
    median_ratio = float(ratio.median())
    candidates = (1.0, 10_000.0)
    multiplier = min(candidates, key=lambda value: abs(np.log10(median_ratio / value)))
    if abs(np.log10(median_ratio / multiplier)) > np.log10(2.0):
        raise ValueError(f"未知成交额单位：median ratio={median_ratio:.6g}")
    return multiplier, median_ratio


def apply_date_filter(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame
    if DATE_FROM is not None:
        out = out[out["trading_date"] >= pd.Timestamp(DATE_FROM)]
    if DATE_TO is not None:
        out = out[out["trading_date"] <= pd.Timestamp(DATE_TO)]
    return out.reset_index(drop=True)


def load_duckdb_minute() -> pd.DataFrame:
    if not DB_TABLE.replace("_", "").isalnum():
        raise ValueError(f"非法 DB_TABLE={DB_TABLE!r}")
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        available = {row[0] for row in con.execute("SHOW TABLES").fetchall()}
        if DB_TABLE not in available:
            raise ValueError(f"DuckDB 不存在表 {DB_TABLE}; available={sorted(available)}")
        frame = con.execute(
            f'''
            SELECT bar_time AS datetime,
                   trade_date AS trading_date,
                   main_contract AS contract,
                   open, high, low, close,
                   amt AS amount_raw,
                   volume,
                   oi,
                   data_source
            FROM {DB_TABLE}
            ORDER BY bar_time
            '''
        ).fetchdf()
    finally:
        con.close()
    frame["source"] = "duckdb"
    return frame


def load_csv_minute() -> pd.DataFrame:
    frame = pd.read_csv(
        CSV_PATH,
        usecols=[
            "datetime", "trading_date", "dominant_id",
            "open", "high", "low", "close",
            "total_turnover", "volume", "open_interest",
        ],
        parse_dates=["datetime", "trading_date"],
    )
    return frame.rename(
        columns={
            "dominant_id": "contract",
            "total_turnover": "amount_raw",
            "open_interest": "oi",
        }
    ).assign(data_source="csv_file", source="csv")


def normalize_minute(frame: pd.DataFrame) -> tuple[pd.DataFrame, float, float]:
    out = frame.copy()
    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
    out["trading_date"] = pd.to_datetime(out["trading_date"], errors="coerce").dt.normalize()
    out["contract"] = normalize_contract(out["contract"])
    numeric = ["open", "high", "low", "close", "amount_raw", "volume", "oi"]
    out[numeric] = out[numeric].apply(pd.to_numeric, errors="coerce")
    if out[["datetime", "trading_date"]].isna().any().any():
        raise ValueError(f"{out['source'].iloc[0]} 存在无法解析的日期")
    if out.duplicated("datetime").any():
        examples = out.loc[out.duplicated("datetime", keep=False), "datetime"].head().tolist()
        raise ValueError(f"{out['source'].iloc[0]} 存在重复分钟时间戳，例如 {examples}")
    multiplier, median_ratio = infer_amount_multiplier(out)
    out["bar_vwap"] = out["amount_raw"] / out["volume"].replace(0, np.nan) / multiplier
    out["time"] = out["datetime"].dt.strftime("%H:%M")
    return apply_date_filter(out.sort_values("datetime")), multiplier, median_ratio


db_raw = load_duckdb_minute()
csv_raw = load_csv_minute()
db, db_amount_multiplier, db_amount_ratio = normalize_minute(db_raw)
csv, csv_amount_multiplier, csv_amount_ratio = normalize_minute(csv_raw)

print(
    f"DuckDB amount multiplier={db_amount_multiplier:,.0f}, median ratio={db_amount_ratio:,.3f}; "
    f"CSV amount multiplier={csv_amount_multiplier:,.0f}, median ratio={csv_amount_ratio:,.3f}"
)

## 2. 数据覆盖、共同分钟和逐字段差异


In [ ]:
def dataset_profile(frame: pd.DataFrame, name: str, multiplier: float, ratio: float) -> dict[str, object]:
    return {
        "dataset": name,
        "rows": len(frame),
        "start_datetime": frame["datetime"].min(),
        "end_datetime": frame["datetime"].max(),
        "trading_days": frame["trading_date"].nunique(),
        "contracts": frame["contract"].nunique(),
        "duplicate_datetime": int(frame.duplicated("datetime").sum()),
        "null_numeric_cells": int(frame[["open", "high", "low", "close", "volume", "oi", "bar_vwap"]].isna().sum().sum()),
        "amount_multiplier": multiplier,
        "amount_ohlc_median_ratio": ratio,
    }


dataset_summary = pd.DataFrame(
    [
        dataset_profile(db, "duckdb", db_amount_multiplier, db_amount_ratio),
        dataset_profile(csv, "csv", csv_amount_multiplier, csv_amount_ratio),
    ]
)
display(dataset_summary)

base_columns = [
    "datetime", "trading_date", "contract", "open", "high", "low", "close",
    "volume", "oi", "bar_vwap", "data_source",
]
comparison = db[base_columns].merge(
    csv[base_columns],
    on="datetime",
    how="outer",
    suffixes=("_db", "_csv"),
    indicator=True,
    validate="one_to_one",
)
comparison["presence"] = comparison["_merge"].map(
    {"left_only": "duckdb_only", "right_only": "csv_only", "both": "both"}
).astype("string")
comparison["trading_date"] = comparison["trading_date_db"].combine_first(comparison["trading_date_csv"])
both = comparison["presence"].eq("both")
comparison["trading_date_match"] = both & comparison["trading_date_db"].eq(comparison["trading_date_csv"])
comparison["contract_match"] = both & comparison["contract_db"].eq(comparison["contract_csv"])

tolerances = {
    "open": PRICE_TOL,
    "high": PRICE_TOL,
    "low": PRICE_TOL,
    "close": PRICE_TOL,
    "volume": VOLUME_TOL,
    "oi": OI_TOL,
    "bar_vwap": VWAP_TOL,
}
field_rows = []
for field, tolerance in tolerances.items():
    left = comparison[f"{field}_db"]
    right = comparison[f"{field}_csv"]
    valid = both & left.notna() & right.notna()
    absolute = (left - right).abs()
    match = both & ((left.isna() & right.isna()) | (valid & absolute.le(tolerance)))
    comparison[f"{field}_abs_diff"] = absolute
    comparison[f"{field}_match"] = match
    if field in {"open", "high", "low", "close", "bar_vwap"}:
        comparison[f"{field}_diff_bp"] = (left - right) / right.replace(0, np.nan) * 10_000.0
    values = absolute[valid]
    field_rows.append(
        {
            "field": field,
            "tolerance": tolerance,
            "compared_rows": int(valid.sum()),
            "mismatch_rows": int((both & ~match).sum()),
            "mismatch_rate": float((both & ~match).sum() / max(int(both.sum()), 1)),
            "mean_abs_diff": float(values.mean()),
            "median_abs_diff": float(values.median()),
            "p95_abs_diff": float(values.quantile(0.95)),
            "max_abs_diff": float(values.max()),
            "correlation": float(left[valid].corr(right[valid])) if valid.sum() > 1 else np.nan,
        }
    )

field_summary = pd.DataFrame(field_rows)
value_match_columns = [f"{field}_match" for field in tolerances]
comparison["any_value_mismatch"] = both & ~comparison[value_match_columns].all(axis=1)
comparison["contract_mismatch"] = both & ~comparison["contract_match"]
comparison["trading_date_mismatch"] = both & ~comparison["trading_date_match"]
for field in tolerances:
    comparison[f"{field}_mismatch"] = both & ~comparison[f"{field}_match"]
comparison["any_mismatch"] = (
    ~both
    | (both & ~comparison["trading_date_match"])
    | (both & ~comparison["contract_match"])
    | comparison["any_value_mismatch"]
)

join_summary = pd.DataFrame(
    {
        "metric": [
            "duckdb_only_minutes", "csv_only_minutes", "common_minutes",
            "contract_mismatch_minutes", "trading_date_mismatch_minutes",
            "common_minutes_with_any_value_mismatch",
        ],
        "value": [
            int(comparison["presence"].eq("duckdb_only").sum()),
            int(comparison["presence"].eq("csv_only").sum()),
            int(both.sum()),
            int((both & ~comparison["contract_match"]).sum()),
            int((both & ~comparison["trading_date_match"]).sum()),
            int(comparison["any_value_mismatch"].sum()),
        ],
    }
)
display(join_summary)
display(field_summary)

## 3. 按日和按年定位差异开始出现的位置


In [ ]:
comparison["year"] = comparison["trading_date"].dt.year
daily_diff = (
    comparison.groupby("trading_date", as_index=False)
    .agg(
        duckdb_minutes=("presence", lambda values: int(values.isin(["duckdb_only", "both"]).sum())),
        csv_minutes=("presence", lambda values: int(values.isin(["csv_only", "both"]).sum())),
        common_minutes=("presence", lambda values: int(values.eq("both").sum())),
        duckdb_only_minutes=("presence", lambda values: int(values.eq("duckdb_only").sum())),
        csv_only_minutes=("presence", lambda values: int(values.eq("csv_only").sum())),
        contract_mismatch_minutes=("contract_mismatch", "sum"),
        value_mismatch_minutes=("any_value_mismatch", "sum"),
        close_mismatch_minutes=("close_mismatch", "sum"),
        volume_mismatch_minutes=("volume_mismatch", "sum"),
        oi_mismatch_minutes=("oi_mismatch", "sum"),
        mean_abs_close_diff_bp=("close_diff_bp", lambda values: float(values.abs().mean())),
        max_abs_close_diff_bp=("close_diff_bp", lambda values: float(values.abs().max())),
        max_abs_volume_diff=("volume_abs_diff", "max"),
        max_abs_oi_diff=("oi_abs_diff", "max"),
    )
    .sort_values("trading_date")
)
daily_diff["anomaly_score"] = (
    daily_diff["duckdb_only_minutes"]
    + daily_diff["csv_only_minutes"]
    + daily_diff["contract_mismatch_minutes"]
    + daily_diff["value_mismatch_minutes"]
)

yearly_summary = (
    daily_diff.assign(year=daily_diff["trading_date"].dt.year)
    .groupby("year", as_index=False)
    .agg(
        trading_days=("trading_date", "count"),
        duckdb_only_minutes=("duckdb_only_minutes", "sum"),
        csv_only_minutes=("csv_only_minutes", "sum"),
        contract_mismatch_minutes=("contract_mismatch_minutes", "sum"),
        value_mismatch_minutes=("value_mismatch_minutes", "sum"),
        close_mismatch_minutes=("close_mismatch_minutes", "sum"),
        volume_mismatch_minutes=("volume_mismatch_minutes", "sum"),
        oi_mismatch_minutes=("oi_mismatch_minutes", "sum"),
        mean_abs_close_diff_bp=("mean_abs_close_diff_bp", "mean"),
        max_abs_close_diff_bp=("max_abs_close_diff_bp", "max"),
    )
)
display(yearly_summary)
display(daily_diff.nlargest(30, "anomaly_score"))

coverage_fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.36, 0.32, 0.32],
    vertical_spacing=0.07,
    subplot_titles=("每日分钟数", "缺失分钟数", "共同分钟的最大收盘价差 (bp)"),
)
coverage_fig.add_trace(
    go.Scatter(x=daily_diff["trading_date"], y=daily_diff["duckdb_minutes"], name="DuckDB", mode="lines", line={"color": "#1f77b4"}),
    row=1, col=1,
)
coverage_fig.add_trace(
    go.Scatter(x=daily_diff["trading_date"], y=daily_diff["csv_minutes"], name="CSV", mode="lines", line={"color": "#ff7f0e", "dash": "dot"}),
    row=1, col=1,
)
coverage_fig.add_trace(
    go.Bar(x=daily_diff["trading_date"], y=daily_diff["duckdb_only_minutes"], name="DuckDB only", marker_color="#1f77b4"),
    row=2, col=1,
)
coverage_fig.add_trace(
    go.Bar(x=daily_diff["trading_date"], y=-daily_diff["csv_only_minutes"], name="CSV only", marker_color="#ff7f0e"),
    row=2, col=1,
)
coverage_fig.add_trace(
    go.Scatter(x=daily_diff["trading_date"], y=daily_diff["max_abs_close_diff_bp"], name="max |close diff|", mode="lines", line={"color": "#d62728"}),
    row=3, col=1,
)
coverage_fig.add_hline(y=0, line_dash="dot", line_color="gray", row=2, col=1)
coverage_fig.update_layout(
    title="DuckDB 与 CSV 分钟覆盖和价格差异时间线",
    template="plotly_white",
    height=950,
    hovermode="x unified",
    barmode="relative",
    legend={"orientation": "h", "y": 1.08},
)
coverage_fig.update_yaxes(title_text="bars", row=1, col=1)
coverage_fig.update_yaxes(title_text="bars", row=2, col=1)
coverage_fig.update_yaxes(title_text="bp", row=3, col=1)
coverage_fig.show(config={"responsive": True, "displaylogo": False})

## 4. 信号敏感特征：A/B/C/D 与持仓方向是否翻转


In [ ]:
def segment_subset(group: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    return group[(group["time"] >= start) & (group["time"] <= end)]


def segment_vwap(group: pd.DataFrame, start: str, end: str) -> float:
    sub = segment_subset(group, start, end)
    if sub.empty:
        return np.nan
    weights = sub["volume"].clip(lower=0)
    if float(weights.sum()) > 0:
        return float(np.average(sub["bar_vwap"], weights=weights))
    return float(sub["bar_vwap"].mean())


def segment_oi_metrics(group: pd.DataFrame, start: str, end: str, prefix: str) -> dict[str, float]:
    sub = segment_subset(group, start, end)
    names = ["oi_delta", "seg_vol", "oi_long_open_ratio", "oi_short_open_ratio", "oi_long_close_ratio", "oi_short_close_ratio"]
    if sub.empty:
        return {f"{prefix}_{name}": np.nan for name in names}
    valid = sub.dropna(subset=["dprice", "doi"])
    total_volume = float(valid["volume"].sum())
    result = {
        f"{prefix}_oi_delta": float(sub["oi"].iloc[-1] - sub["oi"].iloc[0]),
        f"{prefix}_seg_vol": float(sub["volume"].sum()),
    }
    if total_volume <= 0:
        result.update({f"{prefix}_{name}": np.nan for name in names[2:]})
        return result
    result[f"{prefix}_oi_long_open_ratio"] = float(valid.loc[(valid["dprice"] > 0) & (valid["doi"] > 0), "volume"].sum() / total_volume)
    result[f"{prefix}_oi_short_open_ratio"] = float(valid.loc[(valid["dprice"] < 0) & (valid["doi"] > 0), "volume"].sum() / total_volume)
    result[f"{prefix}_oi_long_close_ratio"] = float(valid.loc[(valid["dprice"] < 0) & (valid["doi"] < 0), "volume"].sum() / total_volume)
    result[f"{prefix}_oi_short_close_ratio"] = float(valid.loc[(valid["dprice"] > 0) & (valid["doi"] < 0), "volume"].sum() / total_volume)
    return result


def add_signal_states(features: pd.DataFrame) -> pd.DataFrame:
    out = features.copy()
    long_reversal = (out["B_vwap"] < out["A_vwap"]) & (out["D_vwap"] > out["C_vwap"])
    short_reversal = (out["B_vwap"] > out["A_vwap"]) & (out["D_vwap"] < out["C_vwap"])
    out["reversal_dir"] = np.select([long_reversal, short_reversal], [1, -1], default=0).astype(int)
    out["rev_class"] = np.select(
        [
            long_reversal & (out["C_vwap"] <= out["B_vwap"]) | short_reversal & (out["C_vwap"] >= out["B_vwap"]),
            long_reversal & (out["C_vwap"] > out["B_vwap"]) | short_reversal & (out["C_vwap"] < out["B_vwap"]),
        ],
        ["jia", "yi"],
        default="none",
    )
    ranks = out[["A_vwap", "B_vwap", "C_vwap", "D_vwap"]]
    out["D_rank"] = np.where(
        out["reversal_dir"] > 0,
        ranks.rank(axis=1, method="min", ascending=False)["D_vwap"],
        np.where(out["reversal_dir"] < 0, ranks.rank(axis=1, method="min", ascending=True)["D_vwap"], np.nan),
    )
    out["rev_grade"] = "none"
    out.loc[(out["rev_class"] == "jia") & (out["D_rank"] == 1), "rev_grade"] = "jia1"
    out.loc[(out["rev_class"] == "jia") & (out["D_rank"] == 2), "rev_grade"] = "jia2"
    out.loc[(out["rev_class"] == "jia") & (out["D_rank"] == 3), "rev_grade"] = "jia3"
    out.loc[(out["rev_class"] == "yi") & (out["D_rank"] == 1), "rev_grade"] = "yi1"
    out.loc[(out["rev_class"] == "yi") & (out["D_rank"] >= 2), "rev_grade"] = "yi2"
    out["d_consistency"] = np.where(
        out["reversal_dir"] < 0,
        (out["D1_vwap"] < out["C_vwap"]) & (out["D2_vwap"] < out["D1_vwap"]) & (out["D3_vwap"] < out["D2_vwap"]),
        np.where(
            out["reversal_dir"] > 0,
            (out["D1_vwap"] > out["C_vwap"]) & (out["D2_vwap"] > out["D1_vwap"]) & (out["D3_vwap"] > out["D2_vwap"]),
            False,
        ),
    ).astype(bool)
    out["abc_oi_mean"] = out[["A_oi_delta", "B_oi_delta", "C_oi_delta"]].mean(axis=1)
    out["close_oi_accel"] = out["D_oi_delta"] - out["abc_oi_mean"]
    out["close_oi_accel_sign"] = np.select(
        [out["close_oi_accel"] < 0, out["close_oi_accel"] > 0],
        ["neg", "pos"],
        default="zero",
    )
    ratio_columns = [
        "D_oi_long_open_ratio", "D_oi_short_open_ratio",
        "D_oi_long_close_ratio", "D_oi_short_close_ratio",
    ]
    ratio_map = {
        "D_oi_long_open_ratio": "lo",
        "D_oi_short_open_ratio": "so",
        "D_oi_long_close_ratio": "lc",
        "D_oi_short_close_ratio": "sc",
    }
    out["close_oi_dominant"] = "na"
    valid_ratios = out[ratio_columns].notna().any(axis=1)
    out.loc[valid_ratios, "close_oi_dominant"] = (
        out.loc[valid_ratios, ratio_columns].idxmax(axis=1).map(ratio_map)
    )
    return out


def build_signal_features(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for trading_date, raw_group in frame.groupby("trading_date", sort=True):
        group = raw_group.sort_values("datetime").copy()
        group["dprice"] = group["close"].diff()
        group["doi"] = group["oi"].diff()
        row = {"trading_date": trading_date, "minute_count": len(group), "contract": group["contract"].iloc[0]}
        for segment in ["A", "B", "C", "D", "D1", "D2", "D3"]:
            row[f"{segment}_vwap"] = segment_vwap(group, *SEGMENTS[segment])
            row.update(segment_oi_metrics(group, *SEGMENTS[segment], segment))
        open5 = SEGMENTS["open5_post"] if trading_date >= POST_OPEN_SWITCH else SEGMENTS["open5_pre"]
        row["open5_vwap"] = segment_vwap(group, *open5)
        row["last2_vwap"] = segment_vwap(group, *SEGMENTS["last2"])
        d_sub = segment_subset(group, *SEGMENTS["D"])
        row["D_high"] = float(d_sub["high"].max()) if not d_sub.empty else np.nan
        row["D_low"] = float(d_sub["low"].min()) if not d_sub.empty else np.nan
        row["day_oi_delta"] = float(group["oi"].iloc[-1] - group["oi"].iloc[0])
        row["day_vol"] = float(group["volume"].sum())
        rows.append(row)
    return add_signal_states(pd.DataFrame(rows))


common_dates = sorted(set(db["trading_date"]).intersection(csv["trading_date"]))
features_db = build_signal_features(db[db["trading_date"].isin(common_dates)])
features_csv = build_signal_features(csv[csv["trading_date"].isin(common_dates)])
feature_comparison = features_db.merge(
    features_csv,
    on="trading_date",
    how="inner",
    suffixes=("_db", "_csv"),
    validate="one_to_one",
)

state_columns = ["reversal_dir", "rev_grade", "d_consistency", "close_oi_accel_sign", "close_oi_dominant"]
state_flip_rows = []
for state in state_columns:
    flip = feature_comparison[f"{state}_db"].ne(feature_comparison[f"{state}_csv"])
    feature_comparison[f"{state}_flip"] = flip
    state_flip_rows.append(
        {
            "state": state,
            "common_days": len(feature_comparison),
            "flip_days": int(flip.sum()),
            "flip_rate": float(flip.mean()),
        }
    )
state_flip_summary = pd.DataFrame(state_flip_rows)
feature_comparison["any_state_flip"] = feature_comparison[[f"{state}_flip" for state in state_columns]].any(axis=1)

numeric_features = [
    "A_vwap", "B_vwap", "C_vwap", "D_vwap", "D1_vwap", "D2_vwap", "D3_vwap",
    "open5_vwap", "last2_vwap", "D_oi_delta", "day_oi_delta", "day_vol", "close_oi_accel",
]
for field in numeric_features:
    feature_comparison[f"{field}_diff"] = feature_comparison[f"{field}_db"] - feature_comparison[f"{field}_csv"]

state_flips = feature_comparison[feature_comparison["any_state_flip"]].copy()
state_flips = state_flips.merge(
    daily_diff[["trading_date", "duckdb_only_minutes", "csv_only_minutes", "value_mismatch_minutes", "max_abs_close_diff_bp"]],
    on="trading_date",
    how="left",
)
state_flip_view_columns = [
    "trading_date", "contract_db", "contract_csv", "minute_count_db", "minute_count_csv",
    "reversal_dir_db", "reversal_dir_csv", "rev_grade_db", "rev_grade_csv",
    "d_consistency_db", "d_consistency_csv",
    "close_oi_accel_sign_db", "close_oi_accel_sign_csv",
    "close_oi_dominant_db", "close_oi_dominant_csv",
    "A_vwap_diff", "B_vwap_diff", "C_vwap_diff", "D_vwap_diff",
    "D1_vwap_diff", "D2_vwap_diff", "D3_vwap_diff",
    "D_oi_delta_diff", "close_oi_accel_diff",
    "duckdb_only_minutes", "csv_only_minutes", "value_mismatch_minutes", "max_abs_close_diff_bp",
]
state_flip_view = state_flips[state_flip_view_columns].sort_values("trading_date", ascending=False)
display(state_flip_summary)
display(state_flip_view.head(100))

flip_by_year = (
    feature_comparison.assign(year=feature_comparison["trading_date"].dt.year)
    .groupby("year")[[f"{state}_flip" for state in state_columns]]
    .sum()
    .reset_index()
)
flip_fig = go.Figure()
for state in state_columns:
    flip_fig.add_trace(
        go.Bar(x=flip_by_year["year"], y=flip_by_year[f"{state}_flip"], name=state)
    )
flip_fig.update_layout(
    title="信号敏感状态翻转日数量",
    template="plotly_white",
    barmode="group",
    height=520,
    hovermode="x unified",
    yaxis_title="days",
    legend={"orientation": "h", "y": 1.12},
)
flip_fig.show(config={"responsive": True, "displaylogo": False})

## 5. 异常日的分钟路径


In [ ]:
if INSPECT_DATE is None:
    if not state_flips.empty:
        inspect_date = state_flips.sort_values(
            ["value_mismatch_minutes", "max_abs_close_diff_bp"], ascending=False
        )["trading_date"].iloc[0]
    else:
        inspect_date = daily_diff.nlargest(1, "anomaly_score")["trading_date"].iloc[0]
else:
    inspect_date = pd.Timestamp(INSPECT_DATE).normalize()

inspect = comparison[comparison["trading_date"].eq(inspect_date)].sort_values("datetime")
if inspect.empty:
    raise ValueError(f"INSPECT_DATE={inspect_date.date()} 不在任一数据集")

print(f"钻取日期：{inspect_date.date()}")
display(
    inspect.loc[inspect["any_mismatch"], [
        "datetime", "presence", "contract_db", "contract_csv",
        "close_db", "close_csv", "close_diff_bp",
        "volume_db", "volume_csv", "volume_abs_diff",
        "oi_db", "oi_csv", "oi_abs_diff",
        "bar_vwap_db", "bar_vwap_csv", "bar_vwap_diff_bp",
        "data_source_db",
    ]].head(200)
)

inspect_fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.31, 0.19, 0.25, 0.25],
    vertical_spacing=0.06,
    subplot_titles=("Close", "Close 差异 (bp)", "Volume", "Open interest"),
)
for suffix, label, color, dash in [
    ("db", "DuckDB", "#1f77b4", "solid"),
    ("csv", "CSV", "#ff7f0e", "dot"),
]:
    inspect_fig.add_trace(
        go.Scatter(x=inspect["datetime"], y=inspect[f"close_{suffix}"], name=f"Close {label}", mode="lines", line={"color": color, "dash": dash}),
        row=1, col=1,
    )
    inspect_fig.add_trace(
        go.Scatter(x=inspect["datetime"], y=inspect[f"volume_{suffix}"], name=f"Volume {label}", mode="lines", line={"color": color, "dash": dash}),
        row=3, col=1,
    )
    inspect_fig.add_trace(
        go.Scatter(x=inspect["datetime"], y=inspect[f"oi_{suffix}"], name=f"OI {label}", mode="lines", line={"color": color, "dash": dash}),
        row=4, col=1,
    )
inspect_fig.add_trace(
    go.Bar(x=inspect["datetime"], y=inspect["close_diff_bp"], name="Close diff bp", marker_color="#d62728"),
    row=2, col=1,
)
inspect_fig.add_hline(y=0, line_dash="dot", line_color="gray", row=2, col=1)
inspect_fig.update_layout(
    title=f"分钟数据异常日钻取 — {inspect_date.date()}",
    template="plotly_white",
    height=1050,
    hovermode="x unified",
    legend={"orientation": "h", "y": 1.08},
)
inspect_fig.update_yaxes(title_text="price", row=1, col=1)
inspect_fig.update_yaxes(title_text="bp", row=2, col=1)
inspect_fig.update_yaxes(title_text="contracts", row=3, col=1)
inspect_fig.update_yaxes(title_text="contracts", row=4, col=1)
inspect_fig.show(config={"responsive": True, "displaylogo": False})

## 6. 导出审计结果


In [ ]:
mismatch_columns = [
    "datetime", "trading_date", "presence",
    "contract_db", "contract_csv", "data_source_db",
    "open_db", "open_csv", "high_db", "high_csv", "low_db", "low_csv", "close_db", "close_csv",
    "close_diff_bp", "volume_db", "volume_csv", "volume_abs_diff",
    "oi_db", "oi_csv", "oi_abs_diff", "bar_vwap_db", "bar_vwap_csv", "bar_vwap_diff_bp",
    "contract_match", "any_value_mismatch",
]
minute_mismatches = comparison.loc[comparison["any_mismatch"], mismatch_columns].copy()

if EXPORT_RESULTS:
    output_dir = DATA_PIPELINE_REPORT_DIR
    output_dir.mkdir(parents=True, exist_ok=True)
    workbook_path = output_dir / "minute_data_comparison.xlsx"
    detail_path = output_dir / "minute_row_differences.parquet"
    minute_mismatches.to_parquet(detail_path, index=False)
    with pd.ExcelWriter(workbook_path) as writer:
        dataset_summary.to_excel(writer, sheet_name="dataset_summary", index=False)
        join_summary.to_excel(writer, sheet_name="join_summary", index=False)
        field_summary.to_excel(writer, sheet_name="field_summary", index=False)
        yearly_summary.to_excel(writer, sheet_name="yearly_summary", index=False)
        daily_diff.to_excel(writer, sheet_name="daily_diff", index=False)
        state_flip_summary.to_excel(writer, sheet_name="state_flip_summary", index=False)
        state_flip_view.to_excel(writer, sheet_name="state_flip_days", index=False)
    print(f"已输出：{workbook_path}")
    print(f"逐分钟异常：{detail_path} ({len(minute_mismatches):,} rows)")

print("\n最重要的检查顺序：")
print("1) join_summary：是否存在单边缺失分钟")
print("2) yearly_summary：差异从哪一年开始")
print("3) state_flip_days：哪些日期会改变 02 的信号敏感状态")
print("4) 修改 INSPECT_DATE 后重跑第 5 节，查看具体分钟路径")
